# 11 · Physics-Informed DeepONet — Operators Without Labelled Data

**Level:** Research
**Prerequisites:** `09_deeponet`, `07_inverse_parameter_estimation`

Notebook 09 trained a DeepONet on labelled input→output pairs. But generating
labels means solving the PDE many times first — expensive. **Physics-informed
DeepONets** (Wang, Wang & Perdikaris, *Science Advances* 2021) remove that need:
the operator is trained by enforcing the **governing equation on the output
function**, using autograd through the trunk. Little or no labelled solution data
is required.

We demonstrate on the antiderivative operator from notebook 09, but this time with
**zero labelled targets** — only the defining ODE and one boundary condition.

### What you will learn
1. Differentiating the DeepONet output w.r.t. the trunk (query) input
2. Building a physics loss for an *operator*
3. Training with no solution labels at all

### References
- Wang, Wang & Perdikaris (2021), *Learning the solution operator of parametric PDEs with physics-informed DeepONets*, Sci. Adv. 7:eabi8605
- Goswami et al. (2022), *Physics-informed deep neural operator networks* (variational form)


## 1 · The problem, restated as physics

Learn $G:u\mapsto s$ where

$$\frac{d s}{d y}=u(y),\qquad s(0)=0.$$

**Physics loss:** at random query points $y$, enforce
$\partial_y G(u)(y) - u(y)=0$.
**Boundary loss:** $G(u)(0)=0$.
No ground-truth $s$ anywhere.


In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

m = 100
x_grid = np.linspace(0,1,m)

def random_function(n=1, ls=0.2):
    X=x_grid.reshape(-1,1)
    K=np.exp(-(X-X.T)**2/(2*ls**2))+1e-8*np.eye(m)
    L=np.linalg.cholesky(K)
    return (L@np.random.randn(m,n)).T

def antideriv(u):   # ONLY used to check accuracy at the end
    dx=x_grid[1]-x_grid[0]; s=np.zeros_like(u)
    s[:,1:]=np.cumsum((u[:,1:]+u[:,:-1])/2*dx,axis=1); return s


Using device: cpu


In [ ]:
class DeepONet(nn.Module):
    def __init__(self, m, p=50, width=100):
        super().__init__()
        self.branch=nn.Sequential(nn.Linear(m,width),nn.Tanh(),
                                  nn.Linear(width,width),nn.Tanh(),nn.Linear(width,p))
        self.trunk =nn.Sequential(nn.Linear(1,width),nn.Tanh(),
                                  nn.Linear(width,width),nn.Tanh(),nn.Linear(width,p),nn.Tanh())
        self.b0=nn.Parameter(torch.zeros(1))
    def forward(self,u,y):
        return torch.sum(self.branch(u)*self.trunk(y),dim=1,keepdim=True)+self.b0

model=DeepONet(m).to(device)


In [ ]:
# --- Training set: input functions + their sensor values interpolated at query y ---
N_fun = 1500
U = random_function(N_fun)                        # (N,m) input functions
U_t = torch.tensor(U, dtype=torch.float32, device=device)

def sample_batch(bs):
    idx = np.random.randint(0, N_fun, bs)
    u_b = U_t[idx]                                 # (bs, m)
    y   = torch.rand(bs,1,device=device, requires_grad=True)   # query points
    # u(y): linear-interp the input function at y (differentiable in y not needed for RHS)
    yi = (y.detach().cpu().numpy().ravel()*(m-1))
    lo = np.floor(yi).astype(int); lo=np.clip(lo,0,m-2); fr=yi-lo
    u_at_y = U[idx,lo]*(1-fr)+U[idx,lo+1]*fr
    u_at_y = torch.tensor(u_at_y, dtype=torch.float32, device=device).view(-1,1)
    return u_b, y, u_at_y

# boundary batch (y=0)
def boundary_batch(bs):
    idx=np.random.randint(0,N_fun,bs)
    return U_t[idx], torch.zeros(bs,1,device=device)


In [ ]:
opt=torch.optim.Adam(model.parameters(),lr=1e-3)
EPOCHS=10000
for ep in range(EPOCHS):
    opt.zero_grad()
    u_b,y,u_at_y = sample_batch(2000)
    s = model(u_b,y)
    s_y = torch.autograd.grad(s,y,torch.ones_like(s),create_graph=True)[0]
    L_phys = torch.mean((s_y - u_at_y)**2)         # ds/dy = u(y)
    ub,y0 = boundary_batch(500)
    L_bc = torch.mean(model(ub,y0)**2)             # s(0)=0
    loss = L_phys + L_bc
    loss.backward(); opt.step()
    if ep%2000==0:
        print(f"epoch {ep:5d} | loss {loss.item():.3e} | phys {L_phys.item():.2e} bc {L_bc.item():.2e}")
print("done — trained with ZERO solution labels")


In [ ]:
# --- Test against the true antiderivative (which the model never saw) ---
U_test=random_function(3); S_test=antideriv(U_test)
y_full=torch.tensor(x_grid,dtype=torch.float32,device=device).view(-1,1)
fig,ax=plt.subplots(1,3,figsize=(15,4))
for i in range(3):
    br=torch.tensor(np.repeat(U_test[i:i+1],m,axis=0),dtype=torch.float32,device=device)
    with torch.no_grad():
        pred=model(br,y_full).cpu().numpy().ravel()
    err=np.linalg.norm(pred-S_test[i])/np.linalg.norm(S_test[i])
    ax[i].plot(x_grid,S_test[i],'k-',lw=2,label='true ∫u')
    ax[i].plot(x_grid,pred,'r--',lw=2,label='PI-DeepONet')
    ax[i].plot(x_grid,U_test[i],color='gray',alpha=0.5,label='input u')
    ax[i].set_title(f'unseen #{i+1} (L2={err:.2e})'); ax[i].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 2 · Why this matters

Combining operator learning with physics gives the best of both worlds:

- **No expensive label generation** — the PDE supervises training directly.
- **Fast inference** across a whole family of inputs (the operator property).
- **Better extrapolation** than data-only operators, because outputs must obey
  the equation.

This is a leading direction for real-time surrogates, control, and design
optimisation. The same recipe extends to time-dependent PDEs (the trunk simply
includes a time coordinate) and to the Fourier Neural Operator family.